# XÂY DỰNG MÔ HÌNH DỰ ĐOÁN GIÁ
Trong phần trước, ta đã thực hiện phân tích và làm sạch bộ dữ liệu 'Diamonds'. Trong phần này ta sẽ đi xây dựng mô hình dự đoán giá cho bộ dữ liệu này.

## 1. Tiền xử lý dữ liệu

Mặc dù trong phần trước ta đã làm sạch dữ liệu, tuy nhiên có những biến ta không thể đưa trực tiếp vào mô hình được (chẳng hạn như 'cut', 'color', 'clarity') mà ta phải thực hiện mã hóa nó. 

Ngoài ra, ta sẽ không đưa các biến như 'price_per_carat' hay 'carat_bin' vào mô hình:
- 'price_per_carat' là biến gian lận, nó chứa cả thông tin về giá. Ngoài ra biến này chỉ có khi biết chính xác giá bán và trọng lượng của viên kim cương.
- 'carat_bin' vì ta đã có biến 'carat' liên tục thể hiện chính xác trọng lượng của viên kim cương.


In [35]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/diamonds_cleaned.csv')

In [36]:
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z,price_per_carat,carat_bin
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43,1417.391304,"(0.199, 0.35]"
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31,1552.380952,"(0.199, 0.35]"
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31,1421.739130,"(0.199, 0.35]"
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63,1151.724138,"(0.199, 0.35]"
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75,1080.645161,"(0.199, 0.35]"


In [37]:
df.drop(columns=['price_per_carat', 'carat_bin'], inplace=True)

In [38]:
cut_order = {'Fair':0, 'Good':1, 'Very Good':2, 'Premium':3, 'Ideal':4}
df['cut'] = df['cut'].map(cut_order)

color_order = {'J':0, 'I':1, 'H':2, 'G':3, 'F':4, 'E':5, 'D':6}
df['color'] = df['color'].map(color_order)

clarity_order = {'I1':0, 'SI2':1, 'SI1':2, 'VS2':3, 'VS1':4, 'VVS2':5, 'VVS1':6, 'IF':7}
df['clarity'] = df['clarity'].map(clarity_order)

df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,4,5,1,61.5,55.0,326,3.95,3.98,2.43
1,0.21,3,5,2,59.8,61.0,326,3.89,3.84,2.31
2,0.23,1,5,4,56.9,65.0,327,4.05,4.07,2.31
3,0.29,3,1,3,62.4,58.0,334,4.20,4.23,2.63
4,0.31,1,0,1,63.3,58.0,335,4.34,4.35,2.75


## 2. Chia tập dữ liệu train/test

In [39]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [40]:
print('====== Baseline "luôn đoán bằng trung bình" ======')
mean_base = y_train.mean()
print('Giá trung bình của tập train:', mean_base)
mse_base = np.mean((y_test - mean_base)**2)
print('MSE tính trên tập test:', mse_base)
rmse_base = np.sqrt(mse_base)
print('Sai số trung bình của baseline:', rmse_base)

====== Baseline "luôn đoán bằng trung bình" ======
Giá trung bình của tập train: 3946.8140124720776
MSE tính trên tập test: 15323967.836638073
Sai số trung bình của baseline: 3914.5839927938796


## 3. Xây dựng mô hình dự đoán

### 3.1 Linear regression

Theo phân tích trong phần EDA, các biến 'x', 'y', 'z' có tương quan tuyến tính cao với 'carat' nên ta sẽ loại nó trong mô hình hồi quy tuyến tính để tránh hiện tượng đa cộng tuyến. Ngoài ra các biến 'depth' và 'table' có tương quan tuyến tính ~0 với 'price' nên ta cũng loại nó ra khỏi mô hình này.

In [41]:
X_train_linear = X_train.drop(columns=['x', 'y', 'z', 'depth', 'table'])
X_test_linear = X_test.drop(columns=['x', 'y', 'z', 'depth', 'table'])

#### **Linear regression cơ bản**

In [42]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

lin = LinearRegression()
lin.fit(X_train_linear, y_train)

y_pred = lin.predict(X_test_linear)

mse_lin = mean_squared_error(y_test, y_pred)
r2_lin = r2_score(y_test, y_pred)

print('MSE:', mse_lin)
print('RMSE:', np.sqrt(mse_lin))
print('R²:', r2_lin)

MSE: 1459710.615757269
RMSE: 1208.1848433734256
R²: 0.904698838858711


#### **Linear regression khi lấy log(price)**

Theo phân tích ở phần EDA, phân phối của biến target 'price' có hình dạng bị lệch phải. Do đó ta sẽ thử lấy log(price) xem mô hình có cho kết quả tốt hơn không.

In [43]:
y_train_log = np.log(y_train)

lin_log = LinearRegression()
lin_log.fit(X_train_linear, y_train_log)

y_pred_log = lin_log.predict(X_test_linear)
y_pred_dollar = np.exp(y_pred_log)

mse_log = mean_squared_error(y_test, y_pred_dollar)
r2_log = r2_score(y_test, y_pred_dollar)

print('MSE:', mse_log)
print('RMSE:', np.sqrt(mse_log))
print('R²:', r2_log)

MSE: 1866278931.3997722
RMSE: 43200.450592554844
R²: -120.84507480864814


**Nhận xét:** Sau khi lấy log(price) mô hình cho kết quả tệ hơn nhiều, tệ hơn cả baseline luôn đoán bằng giá trung bình rất nhiều.

#### **Linear regression khi lấy log(price) và log(carat)**

Từ phân tích ở phần EDA, không chỉ biến 'price' bị lệch phải mà cả 'carat' cũng lệch. Ngoài ra biểu đồ scatter log-log price-carat cho thấy log(price) và log(carat) có tương quan tuyến tính rất cao. Do đó, nếu ta chỉ lấy log(price) sẽ không đủ và nó có thể phá đi tương quan giữa price và carat. Ta cần phải lấy log cả 2 biến.

In [44]:
y_train_log = np.log(y_train)
X_train_log = X_train_linear.copy()
X_train_log['carat'] = np.log(X_train_log['carat'])
X_test_log = X_test_linear.copy()
X_test_log['carat'] = np.log(X_test_log['carat'])

lin_log2 = LinearRegression()
lin_log2.fit(X_train_log, y_train_log)

y_pred_log2 = lin_log2.predict(X_test_log)
y_pred_dollar2 = np.exp(y_pred_log2)

mse_log2 = mean_squared_error(y_test, y_pred_dollar2)
r2_log2 = r2_score(y_test, y_pred_dollar2)

print('MSE:', mse_log2)
print('RMSE:', np.sqrt(mse_log2))
print('R²:', r2_log2)

MSE: 877039.2273087272
RMSE: 936.5037251974641
R²: 0.9427401186052076


**Nhận xét:** Sau khi lấy log cả 2 biến 'price' và 'carat', mô hình cho kết quả tốt hơn so với linear regression thông thường.

#### **Hệ số hồi quy (coefficients)**

In [45]:
coefs = pd.Series(lin.coef_, index=X_train_linear.columns).sort_values(ascending=False)
print('Intercept:', lin.intercept_)
print(coefs)

Intercept: -6252.498736294294
carat      8806.233807
clarity     527.086964
color       322.088274
cut         158.040645
dtype: float64


In [46]:
coefs_log = pd.Series(lin_log2.coef_, index=X_train_linear.columns).sort_values(ascending=False)
print('Intercept:', lin_log2.intercept_)
print(coefs_log)

Intercept: 7.793652683714464
carat      1.878759
clarity    0.123195
color      0.077935
cut        0.032459
dtype: float64


**Nhận xét:** Hệ số hồi quy của carat trong mô hình hồi quy tuyến tính sau khi lấy log(price) và log(carat) cho thấy **'nếu carat tăng 1% thì price tăng ~1.88%'**. Và vì 1.88 > 1 nên giá tăng nhanh hơn tỉ lệ trọng lượng, khớp với phân tích từ EDA, price/carat chỉ giảm nhiễu từ trọng lượng chứ không khử được hoàn toàn.

### 3.2 Random forest

#### Mô hình & kết quả

In [47]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print('MSE:', mse_rf)
print('RMSE:', np.sqrt(mse_rf))
print('R²:', r2_rf)

MSE: 295646.51380682254
RMSE: 543.7338630311915
R²: 0.9806979165945522


#### Kiểm tra overfitting

In [48]:
print('Train R²:', r2_score(y_train, rf.predict(X_train)))
print('Test  R²:', r2_score(y_test, y_pred_rf))

Train R²: 0.9975174703751243
Test  R²: 0.9806979165945522


**Nhận xét:** Train R² cao hơn Test R² (chênh khoảng 1.7%) cho thấy mô hình bị overfit nhẹ, nhưng chấp nhận được.

#### Feature importance

In [49]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances)

carat      0.479529
y          0.408417
clarity    0.063349
color      0.031594
z          0.005455
x          0.004710
depth      0.003059
table      0.002235
cut        0.001652
dtype: float64


**Nhận xét:** Kết quả feature importance của mô hình khớp với EDA:
- Kích thước áp đảo: carat (0,48) + y (0,41) + x + z ≈ 0,90
- Trong nhóm 4C: clarity (0,063) > color (0,032) > cut (0,0017)

### 3.3 XGBoost

In [50]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6,
                   random_state=42, n_jobs=-1)

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

mse_xgb = mean_squared_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print('MSE:', mse_xgb)
print('RMSE:', np.sqrt(mse_xgb))
print('R²:', r2_xgb)

MSE: 283362.75
RMSE: 532.3182788520417
R²: 0.9814999103546143


In [51]:
print('Train R²:', r2_score(y_train, xgb.predict(X_train)))
print('Test  R²:', r2_score(y_test, y_pred_xgb))

Train R²: 0.9906691312789917
Test  R²: 0.9814999103546143


**Nhận xét:** Kết quả của mô hình XGBoost có phần nhỉnh hơn hơn Random forest (R² tăng 0.08%) và ít bị overfit hơn.

## 4. Tổng kết

- **Bảng tổng hợp:**

    | Mô hình | Sai số trung bình (RMSE) | R² |
    |----|----|----|
    | Linear regression | 1208 | 0.905 |
    | Log-log | 937 | 0.943 |
    | Random forest | 544 | 0.981 |
    | XGBoost | 532 | 0.981 |

- Hai mô hình Random forest và XGBoost cho kết quả tốt trên bộ dữ liệu này, tốt hơn linear regression và log linear. Lí do là vì hai mô hình này là mô hình dạng cây, có thể tự bắt được quan hệ phi tuyến, không cần log và không ngại đa cộng tuyến.

- Kết quả feature importance từ mô hình khớp với kết quả EDA ban đầu, mức độ ảnh hưởng của các biến đến giá bán theo thứ tự là: kích thước → clarity → color → cut.

- Hệ số hồi quy của mô hình log - log cho thấy: price ~ a·(carat)^{1.88}. Tức là giá sẽ tăng nhanh hơn tỉ lệ trọng lượng, nếu trọng lượng tăng 1% thì giá tăng ~1.88%.

- Hạn chế: 
    - Mô hình Random forest bị overfit nhẹ (train 0,9975 vs test 0,98) nhưng XGBoost đã giảm được đáng kể.
    - Importance có thiên lệch, không phải "thước đo thuần khiết". Dùng để đọc thứ hạng, nhưng không đọc con số tuyệt đối quá chắc chắn.
    - Dữ liệu chỉ gồm những viên kim cương có giá trị dưới 19.000$.

- Khuyến nghị: Có thể tinh chỉnh các tham số trong mô hình Random forest và XGBoost để đạt được kết quả tốt hơn và giảm overfit.